# Lab 3: Natural Language Processing 🎉

**Important**: Go to "Change Runtime Type", Select Runtime Type "T4 GPU" and Runtime version "2025.10".

In [ ]:
!nvidia-smi

## **Task 1: Hands-On HuggingFace Local VLMs usage**

We will use Qwen3-VL-2B-Instruct (decoder-only transformer) interactively for visual question-answering.

In [ ]:
!pip install git+https://github.com/huggingface/transformers flash-attn --no-build-isolation

In [ ]:
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor

# TODO: Define Model
model = None
# TODO: Define Processor
min_pixels =  256*28*28
max_pixels = 1280*28*28
processor = None

In [ ]:
!wget https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen-VL/assets/demo.jpeg ./demo.jpeg

# Display the demo image
from IPython.display import Image
Image(filename='./demo.jpeg')

In [ ]:
# TODO: Define your image and question
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": "./demo.jpeg"},
            {"type": "text", "text": None},
        ],
    }
]

In [ ]:
# Tokenization: Convert image and text into tokens
inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt"
)
inputs = inputs.to(model.device)

In [ ]:
# Generation and Decoding: Perform inference and decode back to text
generated_ids = model.generate(**inputs, max_new_tokens=256)
generated_ids_trimmed = [
    out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
]
output_text = processor.batch_decode(
    generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
)
print(output_text)

## **Task 2: Sentiment Classification**

- Task: Classify whether a text has negative (0) or positive (1) sentiment.
- Model: BERT (encoder-only transformer) + A classification head
- Dataset: SST-2 Dataset

In [ ]:
# TODO: Install EasyTrain

!git clone https://github.com/kentselwyn/EasyTrain.git

%cd EasyTrain

!pip install .

In [ ]:
!pip install --upgrade "transformers==4.57.6" "datasets>=3,<6" peft

In [ ]:
import os
from pathlib import Path

import torch
import torch.nn as nn

import math
import matplotlib.pyplot as plt

from transformers import (
    AutoTokenizer,
    BertConfig,
    BertForSequenceClassification,
    get_linear_schedule_with_warmup,
)

from easytrain import BaseTask, Callback, DatasetProvider, StepOutput, Trainer
from easytrain.runtime import build_loaders, seed_everything, write_json

### **Preparation**

In [ ]:
config = {
    "model_name": "google-bert/bert-base-uncased", # Choose from "FacebookAI/roberta-base", "microsoft/deberta-base",
    "seed": 42,
    "epochs": 3,
    "batch_size": 16,
    "lr": 2e-5,
    "max_length": 256,
    "limit": 50000,
    "num_workers": 0,
    "num_threads": 2,
    "grad_clip_norm": 1.0,
    "monitor": "accuracy",
    "monitor_mode": "max",
}

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Seed anything random with the same seed for reproducibility
seed_everything(config["seed"], num_threads=config["num_threads"])

output_dir = Path(".") / "s03_NLP"
output_dir.mkdir(parents=True, exist_ok=True)

### **Dataset Definition**

Three things to be done:
1. Download and Explore SST-2 Dataset from HuggingFace
2. Configure Tokenizer and Collate Function
3. Create SST-2 Dataset

**Downloading and Exploring the SST-2 Dataset from HuggingFace**

In [ ]:
from datasets import load_dataset

class SentimentData(DatasetProvider):
    def __init__(self, datasets, collator):
        self.datasets = datasets # Must contain "train", "val", and (optionally) "test"
        self.collator = collator

    def get_dataset(self, split):
        return self.datasets[split]

    @property
    def collate_fn(self):
        return self.collator

In [ ]:
# TODO: Load SST Dataset
raw = load_dataset(None, token=False)
# TODO: Split the train and validation split
split = None

print(f"Number of training examples: {len(split['train'])}")
print(f"Number of testing examples: {len(split['test'])}")

In [ ]:
# TODO: Explore the dataset and know its structure
split['train'][5]

In [ ]:
# Define datasets with "train", "val", and "test"
datasets = {
    "train": split["train"],
    "val": split["test"],
    "test": raw["validation"],
}

**Configuring Tokenizer and Collate Function**

In [ ]:
# TODO: Define tokenizer (each language model has its own tokenizer, they CANNOT be mismatched!)
tokenizer = AutoTokenizer.from_pretrained(config["model_name"], token=False)

# Define collate function
class TextCollator:
    def __init__(self, tokenizer, max_length):
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __call__(self, entries):
        # TODO: Perform tokenization
        batch = dict(
            self.tokenizer(
                None,
            )
        )
        # TODO: Also attach the label
        batch["labels"] = None
        # (Optional): Also show text
        # batch["texts"] = None
        return batch

# Construct the collate function
collator = TextCollator(tokenizer, config["max_length"])

In [ ]:
# TODO: See how tokenization is done
collator([datasets["train"][0], datasets["train"][1]])

**Creating SST-2 Dataset**

In [ ]:
provider = SentimentData(datasets, collator)

# Helper function that returns dataloaders for each split
loaders = build_loaders(provider, config, device, ("train", "val", "test"))

### **Model and Task Definition**

Two things to be done:

1. Define BERT Model for Classification
2. Define the Classification Task

**Defining BERT Model for Classification**

In [ ]:
# HuggingFace provides a convenient interface for fine-tuning!
label_names = {
    0: "negative",
    1: "positive",
}
label_config = {
    "id2label": label_names,
    "label2id": {v: k for k, v in label_names.items()}
}
# TODO: Define BERT
bert = None

class BertClassifier(nn.Module):
    def __init__(self, bert):
        super().__init__()
        self.bert = bert

    def forward(self, **inputs):
        # TODO: Forward to bert
        return None

model = BertClassifier(bert).to(device)

**Defining the Classification Task**

In [ ]:
class SentimentClassificationTask(BaseTask):
    """
    Define four functions regarding the Sentiment Classification Task:

    1. `step`: Forward-step and compute loss
    2. `reset_metrics`: Reset metrics
    3. `update_metrics`: Update metrics
    4. `compute_metrics`: Summarize metrics

    """

    def step(self, model, criterion, batch):
        """
        Forward-step and compute loss, then return in StepOutput object
        """
        labels = batch["labels"]
        logits = model(**{k: v for k, v in batch.items() if k != "labels"})
        loss = criterion(logits, labels)
        # TODO: Complete every entry in StepOutput
        return StepOutput(
            loss=loss,
            loss_dict={"Cross Entropy": loss},
            weight=len(labels),
            pred=logits,
            trgt=labels,
        )

    # Reset metrics
    def reset_metrics(self):
        """
        Confusion table: Two rows (negative and positive) and two columns (negative and positive)
        """
        # TODO: Create an empty confusion table
        self.confusion = None


    def update_metrics(self, output):
        # TODO: Update the confusion table
        self.confusion += None


    def compute_metrics(self):
        # TODO: Calculate accuracy, precision, recall, and F1 based on the confusion table
        table = self.confusion.float()
        accuracy = None
        precision = None
        recall = None
        f1 = None
        return {
            "accuracy": accuracy,
            "precision": precision,
            "recall": recall,
            "f1": f1,
        }

In [ ]:
# Some bincount tech
labels = torch.Tensor([0, 0, 1, 1]).to(torch.long)
predicted = torch.Tensor([0, 0, 0, 1]).to(torch.long)

print(
    torch.bincount(labels * 2 + predicted, minlength=4).reshape(2, 2)
)

### **Training**

Two things to be done:
- Define your optimizer
- Train your model!

In [ ]:
# TODO: Define your optimizer
optimizer = None

total_steps = len(loaders["train"]) * config["epochs"]
warmup_steps = int(total_steps * 0.1)
scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)

In [ ]:
# Helper class to see training progress
class TrainingProgress(Callback):
    def on_before_optimizer_step(self, trainer):
        nn.utils.clip_grad_norm_(
            model.parameters(), config["grad_clip_norm"]
        )

    def on_epoch_end(self, trainer, row):
        train, valid = row["train_metrics"], row["val_metrics"]
        print(
            f"Epoch {row['epoch']}:\n"
            f"\tTrain Loss = {train['loss']}\n"
            f"\tValid loss = {valid['loss']}\n"
            f"\tValid accuracy = {valid['accuracy']}\n"
            f"\tValid precision = {valid['precision']}\n"
            f"\tValid recall = {valid['recall']}\n"
            f"\tValid F1 = {valid['f1']}\n"
        )

In [ ]:
# TODO: Train your model
trainer = Trainer(
    model=None,
    task=None,
    optimizer=None,
    criterion=None,
    device=device,
    callbacks=[TrainingProgress()],
    monitor=config["monitor"],
    monitor_mode=config["monitor_mode"],
    scheduler=None,
    scheduler_interval="step",
    output_dir=output_dir,
    config=config,
)
history = trainer.fit(loaders["train"], loaders["val"], epochs=config["epochs"])

In [ ]:
model.eval()
prediction_batch = next(iter(loaders["test"]))
prediction_inputs = {
    key: value.to(device) for key, value in prediction_batch.items() if key != "labels"
}
with torch.no_grad():
    reference_logits = model(**prediction_inputs).cpu()
records = [datasets["test"][i] for i in range(len(prediction_batch["labels"]))]
predictions = [
    {
        "text": row.get("sentence", str(row.get("tokens"))),
        "label": int(prediction_batch["labels"][i]),
        "prediction": int(reference_logits[i].argmax()),
        "positive_probability": float(reference_logits[i].softmax(dim=0)[1]),
    }
    for i, row in enumerate(records)
]
write_json(output_dir / "predictions.json", predictions)
for row in predictions[:5]:
    print(row)

export_dir = output_dir / "bert"
model.bert.save_pretrained(export_dir)
if tokenizer is not None:
    tokenizer.save_pretrained(export_dir)
print("Saved artifacts:", sorted(path.name for path in output_dir.iterdir()))

In [ ]:
# TODO: Try out the trained model!
sentences = [
    "I loved this movie!",
    "The story was dull and disappointing."
]

encoded = tokenizer(
    sentences,
    padding=True,
    truncation=True,
    max_length=config["max_length"],
    return_tensors="pt",
)

with torch.no_grad():
    probabilities = model(**{key: value.to(device) for key, value in encoded.items()}).softmax(1).cpu()

for sentence, probability in zip(sentences, probabilities):
    print(
        {
            "text": sentence,
            "sentiment": label_names[int(probability.argmax())],
            "positive_probability": float(probability[1])
        }
    )